###🚀 STEP 1 — Load Dataset

In [3]:
!pip3 install datasets
!pip3 install scikit-learn

  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached datasets-4.8.4-py3-none-any.whl (526 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [datasets]━━ 1/2 [datasets]
  Using cached scikit_learn-1.8.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (11 kB)
Using cached scikit_learn-1.8.0-cp312-cp312-macosx_12_0_arm64.whl (8.1 MB)


In [4]:
import pandas as pd
from datasets import load_dataset

# load dataset from https://huggingface.co/datasets/tuetschek/atis
dataset = load_dataset("tuetschek/atis")
dataset.set_format(type="pandas")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|███████| 893/893 [00:00<00:00, 230507.32 examples/s]


### 🚀 STEP 2 — Dataset-Based Split (train-valid-test)

In [5]:
import pandas as pd
from datasets import load_dataset

# we don't include any multi-label instances in order to simplify this classification

ATIS_INTENT_MAPPING = {
    'abbreviation': "Abbreviation and Fare Code Meaning Inquiry",
    'aircraft': "Aircraft Type Inquiry",
    'airfare': "Airfare and Fares Questions",
    'airline': "Airline Information Request",
    'airport': "Airport Information and Queries",
    'capacity': "Aircraft Seating Capacity Inquiry",
    'cheapest': "Cheapest Fare Inquiry",
    'city': "Airport Location Inquiry",
    'distance': "Airport Distance Inquiry",
    'flight': "Flight Booking Request",
    'flight_no': "Flight Number Inquiry",
    'flight_time': "Time Inquiry",
    'ground_fare': "Ground Transportation Cost Inquiry",
    'ground_service': "Ground Transportation Inquiry",
    'ground_service+ground_fare': "Airport Ground Transportation and Cost Query",
    'meal': "Inquiry about In-flight Meals",
    'quantity': "Flight Quantity Inquiry",
    'restriction': "Flight Restriction Inquiry"
}

def atis_convert_old_label_to_class_atis(old_label: str):
    if old_label in ATIS_INTENT_MAPPING:
        return ATIS_INTENT_MAPPING[old_label]
    return None


dataset = load_dataset("tuetschek/atis")
dataset.set_format(type="pandas")

samples: pd.DataFrame = dataset["train"][:]
df_test: pd.DataFrame = dataset["test"][:]

samples["label"] = samples["intent"].apply(lambda label: atis_convert_old_label_to_class_atis(label))
df_test["label"] = df_test["intent"].apply(lambda label: atis_convert_old_label_to_class_atis(label))
samples.dropna(subset=["label"], inplace=True)
df_test.dropna(subset=["label"], inplace=True)

In [6]:
display(samples.head())
print("Total samples:", len(samples))
print("Number of intents:", samples["intent"].nunique())
print(samples["intent"].value_counts())

,id,intent,text,slots,label
0,0,flight,i want to fly from boston at 838 am and arrive...,O O O O O B-fromloc.city_name O B-depart_time....,Flight Booking Request
1,1,flight,what flights are available from pittsburgh to ...,O O O O O B-fromloc.city_name O B-toloc.city_n...,Flight Booking Request
2,2,flight_time,what is the arrival time in san francisco for ...,O O O B-flight_time I-flight_time O B-fromloc....,Time Inquiry
3,3,airfare,cheapest airfare from tacoma to orlando,B-cost_relative O O B-fromloc.city_name O B-to...,Airfare and Fares Questions
4,4,airfare,round trip fares from pittsburgh to philadelph...,B-round_trip I-round_trip O O B-fromloc.city_n...,Airfare and Fares Questions


Total samples: 4953
Number of intents: 18
intent
flight                        3666
airfare                        423
ground_service                 255
airline                        157
abbreviation                   147
aircraft                        81
flight_time                     54
quantity                        51
airport                         20
distance                        20
city                            19
ground_fare                     18
capacity                        16
flight_no                       12
meal                             6
restriction                      6
ground_service+ground_fare       1
cheapest                         1
Name: count, dtype: int64


In [7]:
from sklearn.model_selection import train_test_split

# Split samples to train and valid dataset
df_train, df_valid = train_test_split(samples, test_size=0.15, random_state=42)

In [8]:
display(df_train.head())
print("Total train samples:", len(df_train))
print("Number of intents:", df_train["intent"].nunique())
print(df_train["intent"].value_counts())

,id,intent,text,slots,label
1725,1725,abbreviation,what 's the difference between fare code q and...,O O O O O O O B-fare_basis_code O O O B-fare_b...,Abbreviation and Fare Code Meaning Inquiry
4567,4567,airline,what airline is ea the abbreviation for,O O O B-airline_code O O O,Airline Information Request
4818,4818,flight,show me the flights out of love field,O O O O O O B-fromloc.airport_name I-fromloc.a...,Flight Booking Request
2292,2292,flight,which flights depart los angeles destination c...,O O O B-fromloc.city_name I-fromloc.city_name ...,Flight Booking Request
1625,1625,flight,and flights leaving from atlanta to oakland le...,O O O O B-fromloc.city_name O B-toloc.city_nam...,Flight Booking Request


Total train samples: 4210
Number of intents: 18
intent
flight                        3125
airfare                        349
ground_service                 217
abbreviation                   128
airline                        126
aircraft                        71
flight_time                     48
quantity                        47
distance                        17
airport                         16
city                            15
ground_fare                     15
capacity                        14
flight_no                       12
meal                             5
restriction                      3
ground_service+ground_fare       1
cheapest                         1
Name: count, dtype: int64


In [9]:
display(df_valid.head())
print("Total valid samples:", len(df_valid))
print("Number of intents:", df_valid["intent"].nunique())
print(df_valid["intent"].value_counts())

,id,intent,text,slots,label
2535,2535,flight,i need flights that arrive in baltimore from p...,O O O O O O B-toloc.city_name O B-fromloc.city...,Flight Booking Request
635,635,airfare,i would like your rates between atlanta and bo...,O O O O O O B-fromloc.city_name O B-toloc.city...,Airfare and Fares Questions
3533,3533,flight,i would like to book a flight for may twenty s...,O O O O O O O O B-depart_date.month_name B-dep...,Flight Booking Request
2275,2275,ground_service,what types of ground transportation are there ...,O O O O O O O O B-toloc.airport_name I-toloc.a...,Ground Transportation Inquiry
3250,3250,flight,do you have a night flight from washington to ...,O O O O B-depart_time.period_of_day O O B-from...,Flight Booking Request


Total valid samples: 743
Number of intents: 15
intent
flight            541
airfare            74
ground_service     38
airline            31
abbreviation       19
aircraft           10
flight_time         6
airport             4
quantity            4
city                4
distance            3
restriction         3
ground_fare         3
capacity            2
meal                1
Name: count, dtype: int64


In [10]:
display(df_test.head())
print("Total test samples:", len(df_test))
print("Number of intents:", df_test["intent"].nunique())
print(df_test["intent"].value_counts())

,id,intent,text,slots,label
0,0,flight,i would like to find a flight from charlotte t...,O O O O O O O O B-fromloc.city_name O B-toloc....,Flight Booking Request
1,1,airfare,on april first i need a ticket from tacoma to ...,O B-depart_date.month_name B-depart_date.day_n...,Airfare and Fares Questions
2,2,flight,on april first i need a flight going from phoe...,O B-depart_date.month_name B-depart_date.day_n...,Flight Booking Request
3,3,flight,i would like a flight traveling one way from p...,O O O O O O B-round_trip I-round_trip O B-from...,Flight Booking Request
4,4,flight,i would like a flight from orlando to salt lak...,O O O O O O B-fromloc.city_name O B-toloc.city...,Flight Booking Request


Total test samples: 876
Number of intents: 15
intent
flight            632
airfare            48
airline            38
ground_service     36
abbreviation       33
capacity           21
airport            18
distance           10
aircraft            9
flight_no           8
ground_fare         7
meal                6
city                6
quantity            3
flight_time         1
Name: count, dtype: int64


### 🚀 STEP 3 — Baseline Model (TF-IDF + Logistic Regression)

In [11]:
# 3.1 Prepare Data

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

In [12]:
# Encode labels
le = LabelEncoder()
df_train["label"] = le.fit_transform(df_train["intent"])
df_valid["label"] = le.transform(df_valid["intent"])
df_test["label"] = le.transform(df_test["intent"])

print(f"le.classes_: {le.classes_}")
print(f"le.transform(le.classes_) : {le.transform(le.classes_)}")
print(f"train_df: {df_train}")
print(f"val_df: {df_valid}")
print(f"test_df: {df_test}")

le.classes_: ['abbreviation' 'aircraft' 'airfare' 'airline' 'airport' 'capacity'
 'cheapest' 'city' 'distance' 'flight' 'flight_no' 'flight_time'
 'ground_fare' 'ground_service' 'ground_service+ground_fare' 'meal'
 'quantity' 'restriction']
le.transform(le.classes_) : [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17]
train_df:         id        intent                                               text  \
1725  1725  abbreviation  what 's the difference between fare code q and...   
4567  4567       airline            what airline is ea the abbreviation for   
4818  4818        flight              show me the flights out of love field   
2292  2292        flight  which flights depart los angeles destination c...   
1625  1625        flight  and flights leaving from atlanta to oakland le...   
...    ...           ...                                                ...   
4448  4448  abbreviation                                what is fare code h   
466    466        flight  show me

In [37]:
# Term Frequency – Inverse Document Frequency (TF-IDF) Vectorization => Converts text into numerical vectors
vectorizer = TfidfVectorizer(max_features=5000) # Keeps the 5000 most important words in dataset

X_train = vectorizer.fit_transform(df_train["text"])
X_valid = vectorizer.transform(df_valid["text"])
X_test = vectorizer.transform(df_test["text"])

print(f"X_train: {X_train}")
print(f"X_valid: {X_valid}")
print(f"X_test: {X_test}")

y_train = df_train["label"]
y_valid = df_valid["label"]
y_test = df_test["label"]

print(f"y_train: {y_train}")
print(f"y_val: {y_valid}")
print(f"y_test: {y_test}")

X_train: <Compressed Sparse Row sparse matrix of dtype 'float64'
	with 44085 stored elements and shape (4210, 830)>
  Coords	Values
  (0, 806)	0.1409977738687947
  (0, 718)	0.11888686705670362
  (0, 292)	0.500642754331379
  (0, 200)	0.22027168703087738
  (0, 335)	0.4837347905570912
  (0, 246)	0.6335793165508808
  (0, 161)	0.17720231115071108
  (1, 806)	0.17653420144382312
  (1, 718)	0.14885056382197295
  (1, 146)	0.38016322304359523
  (1, 414)	0.22908326351766248
  (1, 308)	0.5879958310940542
  (1, 129)	0.562421396701948
  (1, 356)	0.2960051397180763
  (2, 718)	0.16351864398158095
  (2, 665)	0.21346005696988501
  (2, 490)	0.19797066781933775
  (2, 350)	0.14508340628755784
  (2, 574)	0.5185163831999824
  (2, 551)	0.2971307846813465
  (2, 476)	0.5051281689828134
  (2, 339)	0.5051281689828134
  (3, 350)	0.10550854536907764
  (3, 810)	0.2628139170101054
  (3, 280)	0.35702620810268987
  :	:
  (4206, 741)	0.08201980823532248
  (4206, 137)	0.2659575233762763
  (4206, 592)	0.24950235251841504


In [38]:
vectorizer.get_feature_names_out()

array(['0900', '10', '100', '1000', '1017', '1020', '1024', '1026',
       '1030', '1039', '1045', '1055', '1059', '106', '1083', '11', '110',
       '1100', '1110', '1115', '1130', '1133', '1145', '1158', '12',
       '1200', '1209', '1220', '1222', '1245', '1288', '1291', '130',
       '1300', '137338', '139', '1500', '1505', '163', '1700', '1765',
       '19', '1940', '1991', '1992', '201', '21', '210', '2100', '212',
       '2134', '2153', '217', '225', '229', '230', '257', '269', '270',
       '271', '279', '281', '296', '297', '300', '305', '315', '323',
       '324', '329', '343', '345', '352', '3724', '400', '402', '405',
       '415', '417', '428', '43', '430', '4400', '445', '466', '497766',
       '500', '505', '515', '530', '539', '55', '555', '57', '615', '630',
       '645', '650', '705', '71', '718', '720', '723', '727', '72s',
       '730', '733', '734', '737', '746', '747', '755', '757', '767',
       '771', '80', '813', '815', '819', '82', '823', '825', '838', '845',


### How everything connects
 * Text → TF-IDF → Numbers → Logistic Regression → Predictions
 * Text → vectorizer → X_train
 * Labels → y_train

In [14]:
# 3.2 Train Model
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

preds = clf.predict(X_test)

print(classification_report(y_test, preds, labels=le.transform(le.classes_), target_names=le.classes_))

                            precision    recall  f1-score   support

              abbreviation       1.00      1.00      1.00        33
                  aircraft       0.47      0.89      0.62         9
                   airfare       0.93      0.81      0.87        48
                   airline       1.00      0.74      0.85        38
                   airport       1.00      0.11      0.20        18
                  capacity       1.00      0.33      0.50        21
                  cheapest       0.00      0.00      0.00         0
                      city       0.00      0.00      0.00         6
                  distance       1.00      0.30      0.46        10
                    flight       0.90      0.99      0.94       632
                 flight_no       0.00      0.00      0.00         8
               flight_time       0.00      0.00      0.00         1
               ground_fare       1.00      0.57      0.73         7
            ground_service       0.97      0.92

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behav

In [15]:
# 🎯 Report Macro-F1:

from sklearn.metrics import f1_score

macro_f1 = f1_score(y_test, preds, average="macro")
print("Macro-F1:", macro_f1)

Macro-F1: 0.5045346688117773


In [16]:
# Decode the labels and desplay the result
def decode_predictions(texts,predictions):
    decoded_preds = le.inverse_transform(predictions)
    results_df = pd.DataFrame({
        "text": texts,
        "predicted_intent": decoded_preds
    })

    print(results_df)
    return results_df

In [17]:
X_new = vectorizer.transform(["i need flights that arrive in baltimore from ankara"])

In [18]:
text =  ["i need flights that arrive in baltimore from ankara"]

In [19]:
predictions = clf.predict(X_new)
results_df = decode_predictions(text, predictions)

                                                text predicted_intent
0  i need flights that arrive in baltimore from a...           flight


### 🟢 Step 4 — Save Model and Vectorizer

Since we trained a TF-IDF + Logistic Regression model, we need to save:

  * vectorizer (TF-IDF)

  * classifier (clf)

  * LabelEncoder (le)

In [20]:
!pip3 install joblib

In [21]:
import os
import joblib

In [22]:
url = "./models"

# Create folder if not exists
os.makedirs(url, exist_ok=True)

In [23]:
# Save models
def save_models(le, vectorizer, clf):
    # Save LabelEncoder
    joblib.dump(le, f"{url}/label_encoder.pkl")
    print("LabelEncoder saved.")

    # Save TF-IDF
    joblib.dump(vectorizer, f"{url}/vectorizer.pkl")
    print("TF-IDF saved.")

    # Save classifier
    joblib.dump(clf, f"{url}/clf_model.pkl")
    print("Classifier saved.")

In [24]:
# Load models once
def load_models():
    le = joblib.load(f"{url}/label_encoder.pkl")
    vectorizer = joblib.load(f"{url}/vectorizer.pkl")
    clf = joblib.load(f"{url}/clf_model.pkl")
    return le, vectorizer, clf

In [25]:
def predict_intent(texts):
    if not texts:
        print("error: No text provided")
        return []

    # Transform text
    X = vectorizer.transform(texts)

    # Predict
    preds = clf.predict(X)

    # Decode
    decoded_preds = le.inverse_transform(preds)

    # Return results
    results = [{"text": t, "predicted_intent": p} for t, p in zip(texts, decoded_preds)]
    return results

In [26]:
save_models(le, vectorizer, clf)

LabelEncoder saved.
TF-IDF saved.
Classifier saved.


In [27]:
le, vectorizer, clf = load_models()
print(predict_intent(text))

[{'text': 'i need flights that arrive in baltimore from ankara', 'predicted_intent': 'flight'}]
